# Set up a single-HeMS grid with a rerun, pinned to one version

<div class='alert alert-info'>

In this how-to you set up a grid of single helium stars (single_HeMS)
with the physics of a POSYDON v2.0 rerun, read from POSYDON inlists
pinned to one commit. It follows
[the single-HMS how-to](single_hms_rerun.ipynb); the differences are that
a helium star has no ZAMS model, and is built in several formation steps.

</div>

<div class='alert alert-info'>

The outputs on this page were made with a stand-in for MESA, so the
compilation is instant, and the paths were shortened to `/home/user/...`.
With your own MESA, `posydon-grid setup` compiles MESA, which takes a few
minutes, and the paths are yours.

</div>

## Before you start

You need the same things as in
[Set up an HMS-HMS grid with a POSYDON rerun](hms_hms_rerun.ipynb#Before-you-start):

- POSYDON installed, with `posydon-grid` on your `PATH`;
- `MESA_DIR` and `MESASDK_ROOT` set, and the MESA SDK loaded, for the
  POSYDON version of MESA r11701;
- a clone of POSYDON-MESA-INLISTS on its `v2_sources` branch in
  `~/POSYDON-MESA-INLISTS`.

Make a new, empty folder for this grid and start Jupyter (or a terminal)
there. The cells below run in that folder; a line starting with `!` is a
shell command, which you can also type in a terminal without the `!`.

## 1. Pin the inlists to one commit

As in [the single-HMS how-to](single_hms_rerun.ipynb#1.-Pin-the-inlists-to-one-commit),
use a folder of POSYDON-MESA-INLISTS at a fixed commit. If you made
`~/POSYDON-MESA-INLISTS-115de83` there already, reuse it; otherwise:

```bash
git -C ~/POSYDON-MESA-INLISTS worktree add --detach ~/POSYDON-MESA-INLISTS-115de83 115de83
```

In [1]:
!git -C ~/POSYDON-MESA-INLISTS-115de83 log --oneline -1

115de83 (HEAD, origin/v2_sources, origin/HEAD, v2_sources) Remove the example ini files of r11701/running_scripts


## 2. Choose the rerun

For single helium stars, the v2.0 grids used the rerun
`fix_iniHe_pre_rerun5`:

In [2]:
!grep -A1 used_for ~/POSYDON-MESA-INLISTS-115de83/r11701/releases/v2.0/reruns/fix_iniHe_pre_rerun5/scenario.ini

used_for = CO-HMS_RLO (57), CO-HeMS (50), HMS-HMS (12), single_HeMS (8)



## 3. Write the grid points

A helium star is not started from a ZAMS model you choose: its formation
steps (step 5 below) build it from the mass and metallicity of the grid
point. So the grid points are just `initial_z` and `initial_mass`:

In [3]:
import csv

masses = [2.0, 4.0, 8.0, 16.0, 32.0]
with open("grid.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["initial_z", "initial_mass"])
    for mass in masses:
        writer.writerow([0.0142, mass])

## 4. Write the ini file

The same as for single_HMS, without a ZAMS model:

In [4]:
%%writefile single_HeMS.ini
[grid]
type = single_HeMS
mode = fixed
points = grid.csv
run_directory = run

[scenario]
release = v2.0
rerun = fix_iniHe_pre_rerun5
inlist_repo = ~/POSYDON-MESA-INLISTS-115de83

[output]
history_interval = 1
star1 = history, final_model

[submission]
backend = slurm_array
partition = my_partition
account = my_account
walltime = 2-00:00:00
cpus_per_task = 4

Writing single_HeMS.ini


## 5. The formation steps

The release's `scenario.ini` for single_HeMS says how the star is built.
Each `star1_formation_step<N>` is one MESA run of the star, started from
the model the previous step saved; `star1_formation_every_step` goes
into all of them:

In [5]:
!sed -n '/^\[inlists\]/,/^$/p' ~/POSYDON-MESA-INLISTS-115de83/r11701/releases/v2.0/single_HeMS/scenario.ini

[inlists]
; target = files, applied in order; a later file wins key by key.
; 'binary', 'star1' and 'star2' feed &controls and &star_job
; (&binary_controls and &binary_job); formation steps likewise.
star1_formation_every_step = common/binary/inlist1
star1_formation_step0 = CO-HeMS/star1_formation/inlist_step1
star1_formation_step1 = CO-HeMS/star1_formation/inlist_step2
star1_formation_step2 = single_HeMS/single_star_flag



So there are three steps, the first two the same as for the helium star
of the CO-HeMS binary grids:

- step 0 (`CO-HeMS/star1_formation/inlist_step1`) relaxes a star of the
  given mass to pure helium, and saves it;
- step 1 (`CO-HeMS/star1_formation/inlist_step2`) loads that model and
  gives the helium star the metals of `initial_z`, and saves it;
- step 2 (`single_HeMS/single_star_flag`) loads it and runs the helium
  star to its end.

`posydon-grid explain` shows each step as its own namelists:

In [6]:
!posydon-grid explain single_HeMS.ini > explain.txt
!grep '^[a-z]' explain.txt

star1_formation_step0_controls
star1_formation_step0_job
star1_formation_step1_controls
star1_formation_step1_job
star1_formation_step2_controls
star1_formation_step2_job


## 6. Set up

In [7]:
!posydon-grid setup single_HeMS.ini
!ls run/star1

inlist_step0 inlist_step1 inlist_step2 make         src


The `[output]` options go into the last step only (`inlist_step2`), the
one that runs the star to its end and saves `final_star1.mod`:

In [8]:
!grep -n 'save_model_filename' run/star1/inlist_step*

run/star1/inlist_step0:1684:	save_model_filename = 'initial_star1_step0.mod'
run/star1/inlist_step1:1681:	save_model_filename = 'initial_star1_step1.mod'
run/star1/inlist_step2:1681:	save_model_filename = 'final_star1.mod'


The record shows the rerun's Fortran source, used instead of the
release's for the star:

In [9]:
!grep 'star1/src' run/setup_record.ini
!sed -n '/^\[scenario\]/,/^$/p' run/setup_record.ini

star1/src/run.f = /home/user/MESA/mesa-r11701/star/work/src/run.f
star1/src/run_star_extras.f = r11701/releases/v2.0/reruns/fix_iniHe_pre_rerun5/common/binary/src/run_star_extras.f


[scenario]
release = v2.0
rerun = fix_iniHe_pre_rerun5
mesa_version = r11701
mesa = default
inlist_repo = /home/user/POSYDON-MESA-INLISTS-115de83
inlist_remote = https://github.com/POSYDON-code/POSYDON-MESA-INLISTS.git
inlist_commit = 115de83c33bad5013cb09837fcc679d1d45e816b
inlist_dirty = False
inlist_fingerprint = f822714b620e101161f6eaf37234021966dac7cbe4b04b5720a6ee4ff6c695ce
inlist_files = 17



Submit the grid as in
[the HMS-HMS how-to](hms_hms_rerun.ipynb#6.-Submit-the-grid):
`cd run && ./run_grid.sh`.